# GAT-v4 | Phase C2 — CPTAC RNA (VST) & CNV Processing

Builds `cptac_vst_matrix.csv` and `cptac_cnv_raw.csv` — the two inputs Phase 9
(external validation) reads directly. Run Phase C1 first.

**RNA:** Loads per-patient STAR count files -> raw counts matrix -> PyDESeq2
blind VST (`use_design=False`), same settings as TCGA Phase 2 Cell 9. The
**gene panel is fixed to `ppi_gene_index.csv`** from TCGA training — CPTAC
does **not** run its own DEG discovery; that would break the model's expected
input dimensions and reintroduce a form of leakage (the panel must be an
independent decision from the external cohort).

**CNV:** `data_cna.txt` is already GISTIC2 discrete (-2..2), same convention
as TCGA — no neutral-value conversion needed here (unlike a raw-copy-number
source). We only need `Entrez_Gene_Id -> gene_symbol` mapping, which requires
**one step run on your machine first** (Cell 0 below) since this sandbox
can't reach gene-annotation APIs.

---
| Cell | What |
|------|------|
| 0 | **RUN ON YOUR MACHINE FIRST**: build `entrez_to_symbol.csv` |
| 1 | Imports & paths |
| 2 | Load PPI gene panel (fixed, from TCGA) |
| 3 | Load CPTAC raw STAR counts -> genes x samples matrix |
| 4 | PyDESeq2 blind VST |
| 5 | Reindex RNA to PPI panel -> `cptac_vst_matrix.csv` |
| 6 | CNV: Entrez -> symbol, reindex to PPI panel -> `cptac_cnv_raw.csv` |
| 7 | Audit YAML |


---
## Cell 0 — RUN ON YOUR MACHINE FIRST (not in this notebook's normal flow)

This needs internet access to a gene-annotation service, which the notebook
execution environment may not have depending on where you run it. Run this
once, standalone, in any Python environment with internet + `mygene` installed
(`pip install mygene`). It reads the Entrez IDs straight out of your
`data_cna.txt` and writes `entrez_to_symbol.csv` next to it — Cell 6 below
just loads that CSV, no internet needed at that point.

```python
import pandas as pd
import mygene

cna_path = r"C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\rcc_cptac_gdc\data_cna.txt"
cna = pd.read_csv(cna_path, sep='\t')
entrez_ids = cna['Entrez_Gene_Id'].dropna().astype(int).unique().tolist()
print(f"{len(entrez_ids)} unique Entrez IDs to map")

mg = mygene.MyGeneInfo()
result = mg.querymany(entrez_ids, scopes='entrezgene', fields='symbol', species='human')

mapping = pd.DataFrame(result)
mapping = mapping[~mapping.get('notfound', False).fillna(False)]
mapping = mapping.dropna(subset=['symbol'])[['query', 'symbol']]
mapping.columns = ['entrez_id', 'gene_symbol']
mapping['entrez_id'] = mapping['entrez_id'].astype(int)
mapping = mapping.drop_duplicates('entrez_id')

out_path = r"C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\rcc_cptac_gdc\entrez_to_symbol.csv"
mapping.to_csv(out_path, index=False)
print(f"Mapped {len(mapping)} / {len(entrez_ids)} Entrez IDs -> saved to {out_path}")
```


---
## Cell 1 — Imports & Paths


In [4]:
import sys, time, yaml
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT    = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"           # TCGA processed (for ppi_gene_index.csv)
AUDIT_DIR    = KIRC_ROOT / "audit"

CPTAC_ROOT        = Path(r"C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data")
CPTAC_GENE_EXPR_DIR = CPTAC_ROOT / "RNA" / "gene expression"
CPTAC_CBIOPORTAL_DIR = CPTAC_ROOT / "rcc_cptac_gdc"
CPTAC_CNA_PATH       = CPTAC_CBIOPORTAL_DIR / "data_cna.txt"
CPTAC_ENTREZ_MAP_PATH = CPTAC_CBIOPORTAL_DIR / "entrez_to_symbol.csv"   # from Cell 0

CPTAC_DATA_PROC = KIRC_ROOT / "data" / "cptac_processed"
CPTAC_DATA_PROC.mkdir(parents=True, exist_ok=True)

# Outputs from Phase C1
clin_clean = pd.read_csv(CPTAC_DATA_PROC / 'cptac_clinical_clean.csv')
align_df   = pd.read_csv(CPTAC_DATA_PROC / 'cptac_sample_alignment.csv')
print(f"Clinical patients (Phase C1) : {len(clin_clean)}")
print(f"Aligned tumor samples        : {len(align_df)}")

for p, name in [(CPTAC_GENE_EXPR_DIR, 'RNA gene expression dir'),
                (CPTAC_CNA_PATH, 'data_cna.txt'),
                (CPTAC_ENTREZ_MAP_PATH, 'entrez_to_symbol.csv (run Cell 0 first!)')]:
    print(f"  {'OK  ' if p.exists() else 'MISSING'}  {name:<45} -> {p}")


Clinical patients (Phase C1) : 202
Aligned tumor samples        : 199
  OK    RNA gene expression dir                       -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\RNA\gene expression
  OK    data_cna.txt                                  -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\rcc_cptac_gdc\data_cna.txt
  OK    entrez_to_symbol.csv (run Cell 0 first!)      -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\rcc_cptac_gdc\entrez_to_symbol.csv


---
## Cell 2 — Load PPI Gene Panel (Fixed, From TCGA Training)


In [5]:
gene_idx_df = pd.read_csv(DATA_PROC / 'ppi_gene_index.csv')
ppi_genes = gene_idx_df['gene_symbol'].tolist()
print(f"PPI nodes (fixed panel from TCGA training): {len(ppi_genes):,}")


PPI nodes (fixed panel from TCGA training): 4,414


---
## Cell 3 — Load CPTAC Raw STAR Counts -> Genes x Samples Matrix

Iterates every per-patient folder under `gene expression/`, matches folders to
File IDs via the sample sheet, and only loads samples in `align_df` (tumor)
plus any GDC-listed normal samples (needed so VST can be fit jointly with
normals, same as TCGA Phase 2 — otherwise this step behaves like a single-
condition normalization, which changes the size-factor estimates).


In [6]:
sheet_candidates = list((CPTAC_ROOT / 'RNA').glob('gdc_sample_sheet*.tsv'))
if not sheet_candidates:
    raise FileNotFoundError("gdc_sample_sheet*.tsv not found under CPTAC_ROOT/RNA — check CPTAC_ROOT path.")
sheet = pd.read_csv(sheet_candidates[0], sep='\t')

print(f"Sample sheet rows: {len(sheet)}")

# File ID -> subfolder under gene expression dir (GDC download layout: one folder per File ID)
file_id_to_folder = {}
for sub in CPTAC_GENE_EXPR_DIR.iterdir():
    if sub.is_dir():
        file_id_to_folder[sub.name] = sub

print(f"Per-patient folders found: {len(file_id_to_folder)}")

def read_star_count_file(folder):
    tsvs = list(folder.glob('*.tsv'))
    tsvs = [t for t in tsvs if 'star_gene_counts' in t.name or t.name.endswith('.tsv')]
    if not tsvs:
        return None
    df = pd.read_csv(tsvs[0], sep='\t', comment='#', skiprows=[0])
    # Skip the 4 summary rows (N_unmapped, N_multimapping, N_noFeature, N_ambiguous)
    df = df[~df['gene_id'].astype(str).str.startswith('N_')]
    df['gene_id_clean'] = df['gene_id'].str.split('.').str[0]
    # GDC STAR files list the same base Ensembl ID twice for pseudoautosomal (PAR_Y)
    # genes once the version suffix is stripped -- collapse duplicates WITHIN this file
    # before returning, or the resulting non-unique index breaks downstream map/groupby
    # calls with an "unhashable type: Series" error.
    df = df.groupby('gene_id_clean', as_index=False).agg(
        gene_name=('gene_name', 'first'),
        unstranded=('unstranded', 'sum'))
    return df.set_index('gene_id_clean')[['gene_name', 'unstranded']]

# Which samples to load: all tumor (from align_df) + any normal samples in the sheet
tumor_file_ids = sheet[sheet['File Name'].isin(align_df['rna_file_name'])]['File ID'].tolist()
normal_rows = sheet[sheet['Tissue Type'].str.contains('Normal', na=False)]
normal_file_ids = normal_rows['File ID'].tolist()
all_file_ids = list(dict.fromkeys(tumor_file_ids + normal_file_ids))
print(f"Tumor samples to load  : {len(tumor_file_ids)}")
print(f"Normal samples to load : {len(normal_file_ids)}")

counts_dict = {}
gene_name_map = None
t0 = time.time()
for fid in all_file_ids:
    folder = file_id_to_folder.get(fid)
    if folder is None:
        continue
    df = read_star_count_file(folder)
    if df is None:
        continue
    if gene_name_map is None:
        gene_name_map = df['gene_name']
    counts_dict[fid] = df['unstranded']

counts_raw = pd.DataFrame(counts_dict)
dup_ensembl = counts_raw.index[counts_raw.index.duplicated()]
if len(dup_ensembl) > 0:
    raise ValueError(f"{len(dup_ensembl)} duplicate Ensembl IDs survived per-file dedup "
                      f"(e.g. {list(dup_ensembl[:5])}) -- read_star_count_file()'s PAR_Y "
                      f"collapse should have caught this; investigate before proceeding.")

# gene_name_map is a Series keyed by (deduped) gene_id_clean from the first file loaded.
# .reindex + .fillna is used instead of .map(...get(g,g)) so any symbol lookup miss is a
# clean NaN -> fallback to the Ensembl ID itself, rather than risking a Series being
# returned for a still-duplicated key.
symbols = gene_name_map.reindex(counts_raw.index)
counts_raw.index = symbols.where(symbols.notna(), counts_raw.index)
counts_raw = counts_raw.groupby(counts_raw.index).sum()   # collapse duplicate symbols (different Ensembl IDs, same gene name)
print(f"\nLoaded in {time.time()-t0:.0f}s")
print(f"Raw counts matrix shape (genes x samples): {counts_raw.shape}")

# file_id -> sample type, for metadata
file_id_to_tissue = dict(zip(sheet['File ID'], sheet['Tissue Type']))


Sample sheet rows: 466
Per-patient folders found: 466
Tumor samples to load  : 199
Normal samples to load : 151

Loaded in 72s
Raw counts matrix shape (genes x samples): (59427, 350)


---
## Cell 4 — PyDESeq2 Blind VST

Identical settings to TCGA Phase 2 Cell 9: `dds.vst(use_design=False)`, fit
jointly across tumor+normal, same low-count pre-filter.


In [7]:
from pydeseq2.dds import DeseqDataSet

MIN_COUNTS, MIN_SAMPLES = 10, 10

counts_T = counts_raw.T.fillna(0).clip(lower=0).round(0).astype(int)
counts_T.index.name, counts_T.columns.name = 'sample_id', 'gene_symbol'

keep_mask = (counts_T >= MIN_COUNTS).sum(axis=0) >= MIN_SAMPLES
counts_filt = counts_T.loc[:, keep_mask].copy()
print(f"Genes before/after low-count filter: {counts_T.shape[1]:,} -> {counts_filt.shape[1]:,}")

metadata = pd.DataFrame(
    {'tissue_type': [('tumor' if 'Tumor' in file_id_to_tissue.get(fid, '') else 'normal')
                      for fid in counts_filt.index]},
    index=counts_filt.index)
metadata.index.name = 'sample_id'
print(metadata['tissue_type'].value_counts())

N_CPUS = 4
print(f"\nFitting PyDESeq2 (blind, use_design=False) on {counts_filt.shape[1]:,} genes...")
t0 = time.time()
dds = DeseqDataSet(counts=counts_filt, metadata=metadata, design_factors="tissue_type",
                    ref_level=["tissue_type", "normal"], refit_cooks=True, n_cpus=N_CPUS)
dds.deseq2()
dds.vst(use_design=False)
print(f"Done in {(time.time()-t0)/60:.1f} min")

if 'vst_counts' in dds.layers:
    vst_arr = dds.layers['vst_counts']
else:
    vst_key = [k for k in dds.layers.keys() if 'vst' in k.lower()]
    vst_arr = dds.layers[vst_key[0]]

vst_matrix_full = pd.DataFrame(vst_arr.T, index=counts_filt.columns, columns=counts_filt.index)
print(f"VST matrix shape (genes x samples): {vst_matrix_full.shape}")
print(f"VST value range: {vst_matrix_full.values.min():.2f} - {vst_matrix_full.values.max():.2f}")


Genes before/after low-count filter: 59,427 -> 37,874
tissue_type
tumor     199
normal    151
Name: count, dtype: int64

Fitting PyDESeq2 (blind, use_design=False) on 37,874 genes...
Using None as control genes, passed at DeseqDataSet initialization


C:\Users\jiyad\AppData\Local\Temp\ipykernel_8060\2504348085.py:22: DeprecationWarning: ref_level is deprecated and no longer has any effect. It will beremoved in a future release.
  dds = DeseqDataSet(counts=counts_filt, metadata=metadata, design_factors="tissue_type",
C:\Users\jiyad\AppData\Local\Temp\ipykernel_8060\2504348085.py:22: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_filt, metadata=metadata, design_factors="tissue_type",
Fitting size factors...
... done in 0.48 seconds.

Fitting dispersions...
... done in 14.24 seconds.

Fitting dispersion trend curve...
... done in 0.68 seconds.

Fitting MAP dispersions...
... done in 18.06 seconds.

Fitting LFCs...
... done in 7.29 seconds.

Calculating cook's distance...
... done in 0.83 seconds.

Replacing 2062 outlier genes.

Fitting dispersions...
... done in 0.85 seconds.

Fitting MAP dispers

Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


... done in 0.44 seconds.

Fitting dispersions...
... done in 14.81 seconds.



Done in 1.2 min
VST matrix shape (genes x samples): (37874, 350)
VST value range: 1.88 - 21.39


---
## Cell 5 — Reindex to PPI Panel, Subset to Tumor -> `cptac_vst_matrix.csv`


In [8]:
tumor_file_ids_present = [fid for fid in tumor_file_ids if fid in vst_matrix_full.columns]
vst_tumor = vst_matrix_full[tumor_file_ids_present].copy()

# Rename columns from File ID -> Sample ID (matches cptac_sample_alignment.csv 'sample_id')
file_id_to_sample_id = dict(zip(sheet['File ID'], sheet['Sample ID']))
vst_tumor.columns = [file_id_to_sample_id.get(c, c).split(',')[0].strip() for c in vst_tumor.columns]

vst_ppi = vst_tumor.reindex(ppi_genes)
n_found = vst_ppi.notna().any(axis=1).sum()
print(f"PPI genes found in CPTAC RNA: {n_found:,} / {len(ppi_genes):,} ({n_found/len(ppi_genes)*100:.1f}%)")

out_path = CPTAC_DATA_PROC / 'cptac_vst_matrix.csv'
vst_tumor.to_csv(out_path)   # save full matrix (all genes); Phase 9 reindexes to ppi_genes itself
print(f"Saved -> {out_path}  shape={vst_tumor.shape}")


PPI genes found in CPTAC RNA: 4,208 / 4,414 (95.3%)
Saved -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\cptac_processed\cptac_vst_matrix.csv  shape=(37874, 199)


---
## Cell 6 — CNV: Entrez -> Symbol, Reindex -> `cptac_cnv_raw.csv`

`data_cna.txt` is already GISTIC2 discrete (`-2..2`, neutral=0) — same
convention as TCGA. **Note for Phase 9**: since this is not diploid-centered
raw copy number, set `CPTAC_CNV_NEUTRAL_VALUE = 0` (not `2.0`) when you run
that notebook, or the conversion will incorrectly shift already-correct
values.


In [9]:
cna = pd.read_csv(CPTAC_CNA_PATH, sep='\t')
print(f"CNA matrix shape (rows=genes, cols=Entrez_Gene_Id + samples): {cna.shape}")

entrez_map = pd.read_csv(CPTAC_ENTREZ_MAP_PATH)   # from Cell 0, run on your machine
entrez_to_symbol = dict(zip(entrez_map['entrez_id'], entrez_map['gene_symbol']))

cna['gene_symbol'] = cna['Entrez_Gene_Id'].map(entrez_to_symbol)
n_mapped = cna['gene_symbol'].notna().sum()
print(f"Entrez IDs mapped to symbol: {n_mapped:,} / {len(cna):,} ({n_mapped/len(cna)*100:.1f}%)")

cna_mapped = cna.dropna(subset=['gene_symbol']).drop(columns=['Entrez_Gene_Id'])
cna_mapped = cna_mapped.set_index('gene_symbol')
cna_mapped = cna_mapped.groupby(cna_mapped.index).mean()   # collapse duplicate symbols

# cBioPortal sample IDs (e.g. C3L-00004-01) should already match Sample ID convention
cna_ppi = cna_mapped.reindex(ppi_genes)
n_found_cnv = cna_ppi.notna().any(axis=1).sum()
print(f"PPI genes found in CPTAC CNV: {n_found_cnv:,} / {len(ppi_genes):,} ({n_found_cnv/len(ppi_genes)*100:.1f}%)")

out_path_cnv = CPTAC_DATA_PROC / 'cptac_cnv_raw.csv'
cna_mapped.to_csv(out_path_cnv)   # save full matrix; Phase 9 reindexes itself
print(f"Saved -> {out_path_cnv}  shape={cna_mapped.shape}")

# Sanity: overlap between CNV sample columns and clinical/RNA patients
cnv_patients = set('-'.join(c.split('-')[:2]) for c in cna_mapped.columns)
clin_patients = set(clin_clean['patient_id'])
overlap = cnv_patients & clin_patients
print(f"\nCNV sample columns matching clinical patients: {len(overlap)} / {len(clin_patients)}")
if len(overlap) < 0.8 * len(clin_patients):
    print("WARNING: <80% patient overlap between CNV and clinical — check sample ID format "
          "(e.g. CNV columns may need trimming, e.g. 'C3L-00004-01' -> 'C3L-00004').")


CNA matrix shape (rows=genes, cols=Entrez_Gene_Id + samples): (38275, 354)
Entrez IDs mapped to symbol: 38,245 / 38,275 (99.9%)
PPI genes found in CPTAC CNV: 4,243 / 4,414 (96.1%)
Saved -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\cptac_processed\cptac_cnv_raw.csv  shape=(38217, 353)

CNV sample columns matching clinical patients: 202 / 202


---
## Cell 7 — Audit YAML


In [10]:
audit = {
    'phase': 'Phase C2 — CPTAC RNA (VST) & CNV Processing',
    'timestamp': datetime.now().isoformat(),
    'rna': {
        'n_tumor_samples': len(tumor_file_ids_present),
        'n_normal_samples_used_for_vst_fit': len(normal_file_ids),
        'vst_method': 'PyDESeq2 dds.vst(use_design=False), blind, joint tumor+normal fit',
        'ppi_genes_found_pct': round(float(n_found / len(ppi_genes) * 100), 1),
        'output': str(out_path),
    },
    'cnv': {
        'source': 'cBioPortal data_cna.txt (rcc_cptac_gdc)',
        'convention': 'GISTIC2 discrete (-2..2), neutral=0 — SAME as TCGA, no shift needed',
        'entrez_mapped_pct': round(float(n_mapped / len(cna) * 100), 1),
        'ppi_genes_found_pct': round(float(n_found_cnv / len(ppi_genes) * 100), 1),
        'output': str(out_path_cnv),
    },
    'important_note_for_phase9': 'Set CPTAC_CNV_NEUTRAL_VALUE = 0 in Phase 9 (not 2.0) '
                                  'since this CNV source is already GISTIC2-style centered at 0.',
    'status': 'COMPLETE',
}
audit_path = AUDIT_DIR / 'phaseC2_cptac_rna_cnv_processing.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")
print("\nPhase C2 complete. Ready for Phase 9 external validation")
print("(remember: CPTAC_CNV_NEUTRAL_VALUE = 0 in Phase 9's Cell 1 config).")


Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phaseC2_cptac_rna_cnv_processing.yaml

Phase C2 complete. Ready for Phase 9 external validation
(remember: CPTAC_CNV_NEUTRAL_VALUE = 0 in Phase 9's Cell 1 config).
